# 04 · Análisis

ANOVA de dos factores con réplicas, regresión del desempeño vs `r` (rendimientos decrecientes), gráficos de desempeño/VRAM/tiempo/costo y comparación de costo contra una API.

In [ ]:
# === Setup en Google Colab (ejecutar solo en Colab) ===
# Descomenta estas líneas la primera vez en Colab.
# Rama con los splits que incluyen los datos manuales (cambiar a main cuando se fusione):
# !git clone -b splits-con-manual https://github.com/Hammonne/triaje-clinico-lora.git
# %cd triaje-clinico-lora
# Colab trae torchao 0.10, que peft 0.21 rechaza (pide >= 0.16) y no usamos:
# !pip uninstall -q -y torchao
# Colab ya trae todas las dependencias; estas líneas solo verifican mínimos e instalan el paquete:
# !pip install -q -r requirements.txt
# !pip install -q -e .
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
# === Celda de configuración (única fuente de parámetros) ===
import sys; sys.path.insert(0, "src")
from pathlib import Path
from triaje.config import Config, set_seed

config = Config()
# En Colab, apuntar las salidas a Drive para sobrevivir desconexiones:
# config.drive_dir = Path("/content/drive/MyDrive/triaje")
# config.results_dir = config.drive_dir / "results"
# config.output_dir  = config.drive_dir / "adaptadores"
set_seed(config.seed)
print("run_id ejemplo:", config.run_id(), "| modelo:", config.model_name)

## Cargar resultados

In [ ]:
from triaje import analisis as A
df = A.cargar_resultados(config.results_dir / "resultados.csv")
df.head()

## ANOVA de dos factores (factores: r y alpha/r; réplicas: semillas)

In [ ]:
print("LoRA:\n", A.anova_dos_factores(df, "lora").round(4))
# print("DoRA:\n", A.anova_dos_factores(df, "dora").round(4))

## Regresión del desempeño vs r y rendimientos decrecientes

In [ ]:
reg = A.regresion_desempeno_vs_r(df, "lora")
print("R2 por modelo:", reg.r2)
print("Mejor modelo:", reg.mejor_modelo)
print("r de rendimientos decrecientes (aprox):", reg.r_rendimientos_decrecientes)

## Verificación fuera del dominio (extremos r = 2 y r = 128)

La regresión se ajusta solo con el factorial (r ∈ {4, 16, 64}). Los extremos sirven para comprobar si lo que predice se cumple fuera de ese rango.

In [ ]:
import numpy as np
extremos = df[(df["tecnica"] == "lora") & (df["r"].isin([2, 128]))]
real = extremos.groupby("r")[A.METRICA_DEFECTO].agg(["mean", "std"])
a, b = reg.coef_log["a"], reg.coef_log["b"]
real["prediccion_log"] = a * np.log(real.index.astype(float)) + b
real

## LoRA vs DoRA (r=16, α/r=2, mismas semillas)

In [ ]:
comp = df[(df["r"] == 16) & (df["alpha"] == 32) & (df["tecnica"].isin(["lora", "dora"]))]
cols = ["esquema_ok_post", "exact_match_categoricos", "f1_prioridad", "tasa_subtriaje",
        "segundos_entrenamiento", "vram_entrenamiento_gb", "segundos_por_ejemplo", "parametros_entrenables"]
comp.groupby("tecnica")[cols].agg(["mean", "std"]).T

## Gráficos (desempeño, VRAM, tiempo, costo, sub-triaje)

In [ ]:
rutas = A.generar_todos_los_graficos(df, config.results_dir)
for r in rutas: print(r)

## Comparación de costo: API de pago vs. nuestro LoRA

In [ ]:
cmp = A.comparar_costo_api(df, n_test=len(df), tokens_in_prom=250, tokens_out_prom=120,
                           precio_in_1m=config.precio_api_usd_por_1m_tokens_in,
                           precio_out_1m=config.precio_api_usd_por_1m_tokens_out)
print(cmp)